# Assignment 06 – Clustering with K-means
**Domain:** Cybersecurity – Clinic Security Monitoring  
**Dataset:** UCI PhiUSIIL Phishing URL dataset.

This is an unsupervised-learning exercise. Clustering does not use the phishing label to form groups; it explores whether URL-structure features form interpretable groups.

## 1. Load dataset and choose numeric features
Choose three available numeric URL-structure columns. The label is deliberately excluded from clustering.

In [ ]:
# Load the official UCI PhiUSIIL phishing URL dataset.
# UCI label convention: 1 = legitimate, 0 = phishing.
!pip -q install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=967)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.columns = [str(c).strip() for c in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label column candidates:", [c for c in df.columns if c.lower() == "label"])
display(df.head())

preferred_cluster = ["URLLength", "DomainLength", "NoOfSubDomain"]
available = {c.lower(): c for c in df.columns}
cluster_features = [available[c.lower()] for c in preferred_cluster if c.lower() in available]
if len(cluster_features) < 2:
    cluster_features = [
        c for c in df.select_dtypes(include=np.number).columns
        if c.lower() not in {"label", "urlsimilarityindex", "urltitlematchscore", "urlcharprob", "tldlegitimateprob"}
    ][:3]
if len(cluster_features) < 2:
    raise ValueError("At least two numeric clustering features are required.")

cluster_data = df[cluster_features].apply(pd.to_numeric, errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median())
print("Clustering features:", cluster_features)
display(cluster_data.describe())

## 2. Scale features before K-means
K-means uses distance. Scaling prevents a feature with a larger numeric range from dominating the clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Use a reproducible sample for fast clustering and visualisation.
sample_n = min(12000, len(cluster_data))
cluster_sample = cluster_data.sample(n=sample_n, random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_sample)
print("Scaled matrix shape:", X_scaled.shape)
print("Scaled feature means (approximately zero):", X_scaled.mean(axis=0).round(3))

## 3. Run K-means with k=3 and plot clusters
The assignment specifically asks for a k=3 run and a plot coloured by cluster.

In [ ]:
kmeans3 = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_ids = kmeans3.fit_predict(X_scaled)

plt.figure(figsize=(8, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=cluster_ids, alpha=0.45)
plt.title("K-means Clusters (k=3)")
plt.xlabel(f"Scaled {cluster_features[0]}")
plt.ylabel(f"Scaled {cluster_features[1]}")
plt.tight_layout()
plt.show()

## 4. Elbow method: test k from 2 to 8
Plot inertia for each value of k. Choose the elbow where the reduction in inertia begins to level off; do not select k solely because it is the default.

In [ ]:
k_values = list(range(2, 9))
inertias = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.title("Elbow Plot for K-means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"k": k_values, "inertia": inertias}))
print("Inspect the curve and state the k where the decrease starts to flatten. The required k=3 result is shown separately above.")

## 5. Name clusters using feature averages
The cluster labels have no inherent meaning. Use the original-scale feature averages to describe each group. Review the averages and provide human-readable names based on the actual profile.

In [ ]:
clustered = cluster_sample.copy()
clustered["cluster"] = cluster_ids
profiles = clustered.groupby("cluster")[cluster_features].mean().round(2)
print("Mean feature values by cluster:")
display(profiles)

# Create concise, data-derived descriptions for each cluster using the first two selected features.
medians = clustered[cluster_features].median()
for cluster_id, row in profiles.iterrows():
    first_desc = "higher" if row[cluster_features[0]] >= medians[cluster_features[0]] else "lower"
    second_desc = "higher" if row[cluster_features[1]] >= medians[cluster_features[1]] else "lower"
    print(f"Cluster {cluster_id}: {first_desc} {cluster_features[0]} and {second_desc} {cluster_features[1]}.")
print("Use these measured profiles to assign clear names; do not infer malicious intent from a cluster alone.")

## 6. Are the clusters useful in cybersecurity?
Write three sentences after reviewing the feature profiles and elbow plot. Suggested evidence-based framing:

1. The clusters summarise groups of URLs with similar structural characteristics, which can help explore common URL patterns.
2. They are not phishing labels: a cluster may contain both legitimate and phishing examples, so the groups should not be treated as a security decision.
3. Their practical value is exploratory and depends on whether the profiles are stable, interpretable, and useful to analysts; if the groups overlap substantially, they may offer limited operational value.

## Completion checklist
- [ ] Two or three numeric features selected and scaled before clustering.
- [ ] K-means with k=3 run and a cluster-coloured scatter plot drawn.
- [ ] Elbow plot covers k=2 through k=8 and selected k is justified from the curve.
- [ ] Every cluster is named using its feature averages.
- [ ] Three sentences assess whether the clusters are useful in the chosen domain.